In [1]:
"""
Paper Plots Generator
Reads *_results.xlsx files and generates paper-ready figures.
Run this AFTER the experiment notebooks have completed.

Generates:
1. Clean ECE overlay (all models, per dataset)
2. RMSE overlay per outlier type (all models, per dataset)
3. Pinball vs TPL side-by-side RMSE (for ECE paradox figure)
4. Worst-case RMSE bar chart
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
import os

# ── Config ──────────────────────────────────────────────
DATASETS = ["Boston", "Concrete", "Energy", "Kin8nm", "Naval", "Yacht"]
MODELS = ["Pinball", "TPL-tauMAD", "MAQR", "QRF"]
PAPER_NAMES = {"Pinball": "Pinball", "TPL-tauMAD": "TPL", "MAQR": "MAQR", "QRF": "QRF"}
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
COLORS = {
    "Pinball": "#E07B7B",
    "TPL-tauMAD": "#8B5CF6",
    "MAQR": "#8B4513",
    "QRF": "#6B7280",
}
MARKERS = {"Pinball": "o", "TPL-tauMAD": "s", "MAQR": "^", "QRF": "D"}

DATA_DIR = "."  # directory containing *_results.xlsx
PLOTS_DIR = "paper_plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (12, 5),
        "font.size": 11,
        "axes.titlesize": 13,
        "legend.fontsize": 10,
    }
)


# ── Helpers ─────────────────────────────────────────────
def read_sheet(wb, sheet_name):
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    header = rows[0]
    data = {}
    for row in rows[1:]:
        if row[0] is None:
            continue
        vals = {}
        for j, h in enumerate(header[1:], 1):
            if h is not None and isinstance(h, (int, float)):
                vals[float(h)] = row[j]
        data[row[0]] = vals
    return data


def get_quantiles(wb):
    ws = wb["ECE_Clean"]
    header = list(ws.iter_rows(values_only=True))[0]
    return [
        float(h)
        for h in header[1:]
        if h is not None and isinstance(h, (int, float)) and h != "Avg"
    ]


def plot_overlay(quantiles, data_dict, title, ylabel, fname, ylim=None):
    """All models on one plot."""
    x = range(len(quantiles))
    xt = [str(q) for q in quantiles]
    fig, ax = plt.subplots(figsize=(13, 5))
    for m in MODELS:
        if m not in data_dict:
            continue
        vals = [data_dict[m].get(q, np.nan) for q in quantiles]
        ax.plot(
            x,
            vals,
            marker=MARKERS[m],
            color=COLORS[m],
            lw=2,
            markersize=5,
            label=PAPER_NAMES[m],
        )
    ax.set_xticks(list(x))
    ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Quantile τ")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10)
    ax.grid(True, ls="--", alpha=0.5)
    if ylim:
        ax.set_ylim(ylim)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {fname}")


# ── 1. Clean ECE overlay per dataset ────────────────────
print("\n=== Clean ECE overlays ===")
for dn in DATASETS:
    fpath = os.path.join(DATA_DIR, f"{dn}_results.xlsx")
    if not os.path.exists(fpath):
        print(f"  SKIP {dn} (file not found)")
        continue
    wb = openpyxl.load_workbook(fpath)
    quantiles = get_quantiles(wb)
    data = read_sheet(wb, "ECE_Clean")
    plot_overlay(
        quantiles,
        data,
        f"{dn} — Clean ECE per Quantile (All Models)",
        "Empirical Calibration Error",
        os.path.join(PLOTS_DIR, f"{dn}_Clean_ECE_All.png"),
    )


# ── 2. RMSE overlay per outlier type per dataset ────────
print("\n=== RMSE overlays per outlier type ===")
for dn in DATASETS:
    fpath = os.path.join(DATA_DIR, f"{dn}_results.xlsx")
    if not os.path.exists(fpath):
        continue
    wb = openpyxl.load_workbook(fpath)
    quantiles = get_quantiles(wb)
    for ot in OUTLIER_TYPES:
        sn = f"RMSE_{ot}_10"
        if sn not in wb.sheetnames:
            continue
        data = read_sheet(wb, sn)
        plot_overlay(
            quantiles,
            data,
            f"{dn} — RMSE per Quantile ({ot.capitalize()} 10%)",
            "RMSE (clean vs contaminated)",
            os.path.join(PLOTS_DIR, f"{dn}_RMSE_All_{ot}_10.png"),
        )


# ── 3. ECE overlay per outlier type per dataset ─────────
print("\n=== ECE under contamination overlays ===")
for dn in DATASETS:
    fpath = os.path.join(DATA_DIR, f"{dn}_results.xlsx")
    if not os.path.exists(fpath):
        continue
    wb = openpyxl.load_workbook(fpath)
    quantiles = get_quantiles(wb)
    for ot in OUTLIER_TYPES:
        sn = f"ECE_{ot}_10"
        if sn not in wb.sheetnames:
            continue
        data = read_sheet(wb, sn)
        plot_overlay(
            quantiles,
            data,
            f"{dn} — ECE per Quantile ({ot.capitalize()} 10%)",
            "Empirical Calibration Error",
            os.path.join(PLOTS_DIR, f"{dn}_ECE_All_{ot}_10.png"),
        )


# ── 4. Pinball vs TPL side-by-side (paradox figure) ─────
print("\n=== Paradox figures (Pinball vs TPL, Multiply 10%) ===")
for dn in ["Boston", "Concrete", "Yacht"]:
    fpath = os.path.join(DATA_DIR, f"{dn}_results.xlsx")
    if not os.path.exists(fpath):
        continue
    wb = openpyxl.load_workbook(fpath)
    quantiles = get_quantiles(wb)
    rmse = read_sheet(wb, "RMSE_multiply_10")
    x = range(len(quantiles))
    xt = [str(q) for q in quantiles]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    # Pinball
    vals_p = [rmse["Pinball"].get(q, 0) for q in quantiles]
    ax1.errorbar(list(x), vals_p, fmt="o-", color=COLORS["Pinball"], lw=2, markersize=5)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(xt, rotation=45, ha="right", fontsize=7)
    ax1.set_title(f"{dn} — Pinball (Multiply 10%)\nAvg RMSE = {np.mean(vals_p):.1f}")
    ax1.set_xlabel("Quantile τ")
    ax1.set_ylabel("RMSE (clean vs contaminated)")
    ax1.grid(True, ls="--", alpha=0.5)

    # TPL
    vals_t = [rmse["TPL-tauMAD"].get(q, 0) for q in quantiles]
    ax2.errorbar(
        list(x), vals_t, fmt="s-", color=COLORS["TPL-tauMAD"], lw=2, markersize=5
    )
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(xt, rotation=45, ha="right", fontsize=7)
    ax2.set_title(f"{dn} — TPL (Multiply 10%)\nAvg RMSE = {np.mean(vals_t):.1f}")
    ax2.set_xlabel("Quantile τ")
    ax2.grid(True, ls="--", alpha=0.5)

    plt.suptitle(
        f"{dn}: Pinball vs TPL under Multiply 10% Contamination", fontsize=14, y=1.02
    )
    plt.tight_layout()
    fname = os.path.join(PLOTS_DIR, f"{dn}_Paradox_Pinball_vs_TPL_Multiply.png")
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {fname}")


# ── 5. Worst-case RMSE bar chart ────────────────────────
print("\n=== Worst-case RMSE bar chart ===")
worst = {m: [] for m in MODELS}
dataset_labels = []

for dn in DATASETS:
    fpath = os.path.join(DATA_DIR, f"{dn}_results.xlsx")
    if not os.path.exists(fpath):
        continue
    wb = openpyxl.load_workbook(fpath)
    dataset_labels.append(dn)
    for m in MODELS:
        max_rmse = 0
        for ot in OUTLIER_TYPES:
            sn = f"RMSE_{ot}_10"
            if sn not in wb.sheetnames:
                continue
            d = read_sheet(wb, sn)
            if m in d:
                vals = [
                    v
                    for v in d[m].values()
                    if v is not None and isinstance(v, (int, float))
                ]
                avg = np.mean(vals) if vals else 0
                max_rmse = max(max_rmse, avg)
        worst[m].append(max_rmse)

x = np.arange(len(dataset_labels))
width = 0.2
fig, ax = plt.subplots(figsize=(14, 6))

for i, m in enumerate(MODELS):
    bars = ax.bar(
        x + i * width,
        worst[m],
        width,
        label=PAPER_NAMES[m],
        color=COLORS[m],
        alpha=0.85,
    )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(dataset_labels, fontsize=11)
ax.set_ylabel("Worst-case RMSE (max over 4 outlier types)")
ax.set_title("Worst-Case RMSE at 10% Contamination — All Datasets")
ax.legend(fontsize=11)
ax.set_yscale("log")
ax.grid(True, ls="--", alpha=0.3, axis="y")
plt.tight_layout()
fname = os.path.join(PLOTS_DIR, "Worst_Case_RMSE_BarChart.png")
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {fname}")


# ── Summary ─────────────────────────────────────────────
total = len([f for f in os.listdir(PLOTS_DIR) if f.endswith(".png")])
print(f"\n✓ {total} plots saved to {PLOTS_DIR}/")
print(f"\nKey plots for the paper:")
print(f"  Table 4  → {PLOTS_DIR}/<dataset>_Clean_ECE_All.png")
print(f"  Table 5  → {PLOTS_DIR}/<dataset>_Paradox_Pinball_vs_TPL_Multiply.png")
print(f"  Table 6-9  → {PLOTS_DIR}/<dataset>_ECE_All_<type>_10.png")
print(f"  Table 10-13 → {PLOTS_DIR}/<dataset>_RMSE_All_<type>_10.png")
print(f"  Table 14 → {PLOTS_DIR}/Worst_Case_RMSE_BarChart.png")


=== Clean ECE overlays ===
  Saved: paper_plots\Boston_Clean_ECE_All.png
  Saved: paper_plots\Concrete_Clean_ECE_All.png
  Saved: paper_plots\Energy_Clean_ECE_All.png
  Saved: paper_plots\Kin8nm_Clean_ECE_All.png
  Saved: paper_plots\Naval_Clean_ECE_All.png
  Saved: paper_plots\Yacht_Clean_ECE_All.png

=== RMSE overlays per outlier type ===
  Saved: paper_plots\Boston_RMSE_All_gaussian_10.png
  Saved: paper_plots\Boston_RMSE_All_multiply_10.png
  Saved: paper_plots\Boston_RMSE_All_uniform_10.png
  Saved: paper_plots\Boston_RMSE_All_skewed_10.png
  Saved: paper_plots\Concrete_RMSE_All_gaussian_10.png
  Saved: paper_plots\Concrete_RMSE_All_multiply_10.png
  Saved: paper_plots\Concrete_RMSE_All_uniform_10.png
  Saved: paper_plots\Concrete_RMSE_All_skewed_10.png
  Saved: paper_plots\Energy_RMSE_All_gaussian_10.png
  Saved: paper_plots\Energy_RMSE_All_multiply_10.png
  Saved: paper_plots\Energy_RMSE_All_uniform_10.png
  Saved: paper_plots\Energy_RMSE_All_skewed_10.png
  Saved: paper_plots\